In [1]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder \
    .appName("PostgreSQL Connection Example") \
    .config("spark.jars", "jars/postgresql-42.7.4.jar") \
    .getOrCreate()

24/12/15 19:29:47 WARN Utils: Your hostname, Ajay resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/15 19:29:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
24/12/15 19:29:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
jdbc_url = "jdbc:postgresql://localhost:5432/postgres"
connection_properties = {
    "user": "postgres",
    "password": "root",
    "driver": "org.postgresql.Driver"
}

In [6]:
# Read data from PostgreSQL into a PySpark DataFrame
emp_df = spark.read.jdbc(url=jdbc_url, table='Employee', properties=connection_properties)

In [7]:
mgr_df = spark.read.jdbc(url=jdbc_url, table='Employee', properties=connection_properties)

In [8]:
df1_alias = emp_df.alias("emp_df")
df2_alias = mgr_df.alias("mgr_df")

joined_df = df1_alias.join(df2_alias, emp_df.reportsto == mgr_df.employeeid, "inner")

result_df = joined_df.select(
    emp_df.employeeid.alias("empid"),
    mgr_df.employeeid.alias("mgrid"),
    emp_df.firstname,emp_df.lastname,
    mgr_df.firstname,mgr_df.lastname,
    emp_df.title.alias("employee_title"),mgr_df.title.alias("manager_title")
)
result_df.show()

+-----+-----+---------+--------+---------+--------+-------------------+---------------+
|empid|mgrid|firstname|lastname|firstname|lastname|     employee_title|  manager_title|
+-----+-----+---------+--------+---------+--------+-------------------+---------------+
|    2|    1|    Nancy| Edwards|   Andrew|   Adams|      Sales Manager|General Manager|
|    6|    1|  Michael|Mitchell|   Andrew|   Adams|         IT Manager|General Manager|
|    3|    2|     Jane| Peacock|    Nancy| Edwards|Sales Support Agent|  Sales Manager|
|    4|    2| Margaret|    Park|    Nancy| Edwards|Sales Support Agent|  Sales Manager|
|    5|    2|    Steve| Johnson|    Nancy| Edwards|Sales Support Agent|  Sales Manager|
|    7|    6|   Robert|    King|  Michael|Mitchell|           IT Staff|     IT Manager|
|    8|    6|    Laura|Callahan|  Michael|Mitchell|           IT Staff|     IT Manager|
+-----+-----+---------+--------+---------+--------+-------------------+---------------+



In [9]:
# Perform transformations (example: filter employees older than 25)
filtered_df = emp_df.filter(emp_df.title != 'General Manager')
filtered_df.show()

+----------+--------+---------+-------------------+---------+----------+----------+--------------------+----------+-----+-------+----------+-----------------+-----------------+--------------------+
|employeeid|lastname|firstname|              title|reportsto| birthdate|  hiredate|             address|      city|state|country|postalcode|            phone|              fax|               email|
+----------+--------+---------+-------------------+---------+----------+----------+--------------------+----------+-----+-------+----------+-----------------+-----------------+--------------------+
|         2| Edwards|    Nancy|      Sales Manager|        1|1958-12-08|2002-05-01|        825 8 Ave SW|   Calgary|   AB| Canada|   T2P 2T3|+1 (403) 262-3443|+1 (403) 262-3322|nancy@chinookcorp...|
|         3| Peacock|     Jane|Sales Support Agent|        2|1973-08-29|2002-04-01|       1111 6 Ave SW|   Calgary|   AB| Canada|   T2P 5M5|+1 (403) 262-3443|+1 (403) 262-6712|jane@chinookcorp.com|
|         

In [10]:

# Write transformed data back to PostgreSQL (optional)
filtered_df.write.jdbc(
    url=jdbc_url,
    table="filtered_employees",
    mode="overwrite",
    properties=connection_properties
)

print("Filtered data written back to PostgreSQL.")

24/12/15 19:36:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filtered data written back to PostgreSQL.
